In [21]:
# import necessary libraries

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import pickle
from sklearn.metrics import classification_report

In [2]:
# load prepared data
data = np.load("citeseer_prepared.npz", allow_pickle=True)

X = data["X"]
y = data["y"]
train_indices = data["train_indices"]
test_indices = data["test_indices"]
node_ids = data["node_ids"]

print("X shape:", X.shape)
print("Number of classes:", len(np.unique(y)))

X shape: (3312, 3703)
Number of classes: 6


In [4]:
# load neighborhood information

with open("citeseer_neighbors.pkl", "rb") as f:
    neighbors = pickle.load(f)

print("Number of nodes:", len(neighbors))

Number of nodes: 3312


In [5]:
# recreate dataset

node_to_index = {
    node_id: index
    for index, node_id in enumerate(node_ids)
}


def get_node_data(node_id):
    center_index = node_to_index[node_id]

    center_features = X[center_index]

    neighbor_ids = list(neighbors[node_id])

    neighbor_features = np.array([
        X[node_to_index[neighbor_id]]
        for neighbor_id in neighbor_ids
    ])

    label = y[center_index]

    return center_features, neighbor_features, label

In [7]:
# recreate collate function

def citeseer_collate(batch):
    centers = []
    neighbor_features = []
    labels = []

    for center, neighbors, label in batch:
        centers.append(torch.tensor(center, dtype=torch.float32))
        neighbor_features.append(
            torch.tensor(neighbors, dtype=torch.float32)
        )
        labels.append(label)

    centers = torch.stack(centers)
    labels = torch.tensor(labels, dtype=torch.long)

    return centers, neighbor_features, labels

In [8]:
# deep sets model

class CiteseerDeepSets(nn.Module):
    def __init__(self, input_dim=3703, hidden_dim=64, num_classes=6):
        super().__init__()

        # Transform each neighbor independently
        self.phi = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        # Transform the center node
        self.center_embedding = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU()
        )

        # Classifier
        self.rho = nn.Sequential(
            nn.Linear(hidden_dim + hidden_dim, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, center, neighbors):
        # neighbors: [number_of_neighbors, 3703]
        neighbor_features = self.phi(neighbors)

        # Permutation-invariant aggregation
        pooled = neighbor_features.sum(dim=0)

        # Embed center node
        center_features = self.center_embedding(center)

        # Combine center + neighborhood representation
        combined = torch.cat([center_features, pooled], dim=0)

        output = self.rho(combined)

        return output

In [9]:
# create the model

model = CiteseerDeepSets()

print(model)

CiteseerDeepSets(
  (phi): Sequential(
    (0): Linear(in_features=3703, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=64, bias=True)
  )
  (center_embedding): Sequential(
    (0): Linear(in_features=3703, out_features=64, bias=True)
    (1): ReLU()
  )
  (rho): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Linear(in_features=64, out_features=6, bias=True)
  )
)


In [10]:
# check number of parameters
num_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

print("Trainable parameters:", num_parameters)

Trainable parameters: 486918


In [13]:
# test before forward passing
train_dataset = CiteseerDataset(train_indices)
test_dataset = CiteseerDataset(test_indices)

print("Training samples:", len(train_dataset))
print("Test samples:", len(test_dataset))

Training samples: 2649
Test samples: 663


In [14]:
# forward-pass
center, neighbor_features, label = train_dataset[0]

center = torch.tensor(center, dtype=torch.float32)
neighbor_features = torch.tensor(
    neighbor_features,
    dtype=torch.float32
)

output = model(center, neighbor_features)

print("Center shape:", center.shape)
print("Neighbors shape:", neighbor_features.shape)
print("Output shape:", output.shape)
print("Output:", output)
print("True label:", label)

Center shape: torch.Size([3703])
Neighbors shape: torch.Size([1, 3703])
Output shape: torch.Size([6])
Output: tensor([ 0.1094,  0.0558, -0.0664, -0.0931,  0.0523,  0.1089],
       grad_fn=<ViewBackward0>)
True label: 0


In [15]:
# create data loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=citeseer_collate
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    collate_fn=citeseer_collate
)

print("Training batches:", len(train_loader))
print("Test batches:", len(test_loader))

Training batches: 166
Test batches: 42


In [16]:
# loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function:", criterion)
print("Optimizer:", optimizer)

Loss function: CrossEntropyLoss()
Optimizer: Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [17]:
# train for 30 epochs
num_epochs = 30

for epoch in range(num_epochs):
    model.train()

    total_loss = 0.0

    for centers, neighbor_features, labels in train_loader:

        optimizer.zero_grad()

        batch_outputs = []

        for i in range(len(centers)):
            output = model(
                centers[i],
                neighbor_features[i]
            )

            batch_outputs.append(output)

        outputs = torch.stack(batch_outputs)

        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)

    print(
        f"Epoch {epoch + 1:02d}/{num_epochs}, "
        f"Loss: {average_loss:.4f}"
    )

Epoch 01/30, Loss: 1.1604
Epoch 02/30, Loss: 0.4401
Epoch 03/30, Loss: 0.1419
Epoch 04/30, Loss: 0.0409
Epoch 05/30, Loss: 0.0263
Epoch 06/30, Loss: 0.0217
Epoch 07/30, Loss: 0.0144
Epoch 08/30, Loss: 0.0131
Epoch 09/30, Loss: 0.0064
Epoch 10/30, Loss: 0.0068
Epoch 11/30, Loss: 0.0018
Epoch 12/30, Loss: 0.0038
Epoch 13/30, Loss: 0.0028
Epoch 14/30, Loss: 0.0010
Epoch 15/30, Loss: 0.0012
Epoch 16/30, Loss: 0.0032
Epoch 17/30, Loss: 0.0012
Epoch 18/30, Loss: 0.0020
Epoch 19/30, Loss: 0.0009
Epoch 20/30, Loss: 0.0013
Epoch 21/30, Loss: 0.0004
Epoch 22/30, Loss: 0.0002
Epoch 23/30, Loss: 0.0001
Epoch 24/30, Loss: 0.0001
Epoch 25/30, Loss: 0.0001
Epoch 26/30, Loss: 0.0001
Epoch 27/30, Loss: 0.0001
Epoch 28/30, Loss: 0.0001
Epoch 29/30, Loss: 0.0001
Epoch 30/30, Loss: 0.0000


In [18]:
# evaluate test accuracy

model.eval()

correct = 0
total = 0

with torch.no_grad():
    for centers, neighbor_features, labels in test_loader:

        batch_outputs = []

        for i in range(len(centers)):
            output = model(
                centers[i],
                neighbor_features[i]
            )

            batch_outputs.append(output)

        outputs = torch.stack(batch_outputs)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

accuracy = correct / total

print(f"Test accuracy: {accuracy:.4f}")
print(f"Test accuracy: {accuracy * 100:.2f}%")

Test accuracy: 0.7541
Test accuracy: 75.41%


In [20]:
# create confusion matrix
from sklearn.metrics import confusion_matrix

all_predictions = []
all_labels = []

model.eval()

with torch.no_grad():
    for centers, neighbor_features, labels in test_loader:

        batch_outputs = []

        for i in range(len(centers)):
            output = model(
                centers[i],
                neighbor_features[i]
            )

            batch_outputs.append(output)

        outputs = torch.stack(batch_outputs)

        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

cm = confusion_matrix(all_labels, all_predictions)

print("Confusion matrix:")
print(cm)

Confusion matrix:
[[ 24   6   6   2   2  10]
 [  5 101   4   5   1   3]
 [  2   1 113   2  15   7]
 [  1  12   2  78   7   2]
 [  1   2   9   7 102  13]
 [  6   4   5   3  18  82]]


In [22]:
# calculate classification report

class_names = [
    "AI",
    "Agents",
    "DB",
    "HCI",
    "IR",
    "ML"
]

print(
    classification_report(
        all_labels,
        all_predictions,
        target_names=class_names,
        digits=4
    )
)

              precision    recall  f1-score   support

          AI     0.6154    0.4800    0.5393        50
      Agents     0.8016    0.8487    0.8245       119
          DB     0.8129    0.8071    0.8100       140
         HCI     0.8041    0.7647    0.7839       102
          IR     0.7034    0.7612    0.7312       134
          ML     0.7009    0.6949    0.6979       118

    accuracy                         0.7541       663
   macro avg     0.7397    0.7261    0.7311       663
weighted avg     0.7526    0.7541    0.7523       663



In [23]:
# permutation-invariance test
model.eval()

center, neighbors_sample, label = test_dataset[0]

center = torch.tensor(center, dtype=torch.float32)
neighbors_sample = torch.tensor(
    neighbors_sample,
    dtype=torch.float32
)

with torch.no_grad():
    original_output = model(center, neighbors_sample)
    original_probability = torch.softmax(
        original_output, dim=0
    )

    permutation = torch.randperm(
        neighbors_sample.size(0)
    )

    shuffled_neighbors = neighbors_sample[permutation]

    shuffled_output = model(center, shuffled_neighbors)
    shuffled_probability = torch.softmax(
        shuffled_output, dim=0
    )

max_difference = torch.max(
    torch.abs(original_probability - shuffled_probability)
).item()

print("Original probabilities:")
print(original_probability)

print("\nShuffled probabilities:")
print(shuffled_probability)

print("\nMaximum probability difference:", max_difference)

Original probabilities:
tensor([1.0334e-03, 2.3184e-04, 6.5825e-06, 4.0256e-03, 9.9412e-01, 5.7928e-04])

Shuffled probabilities:
tensor([1.0334e-03, 2.3184e-04, 6.5825e-06, 4.0256e-03, 9.9412e-01, 5.7928e-04])

Maximum probability difference: 0.0


In [24]:
# repeat permutation-invariance test for multiple samples
model.eval()

center, neighbors_sample, label = test_dataset[0]

center = torch.tensor(center, dtype=torch.float32)
neighbors_sample = torch.tensor(
    neighbors_sample,
    dtype=torch.float32
)

with torch.no_grad():
    original_output = model(center, neighbors_sample)
    original_probability = torch.softmax(
        original_output, dim=0
    )

    differences = []

    for _ in range(10):
        permutation = torch.randperm(
            neighbors_sample.size(0)
        )

        shuffled_neighbors = neighbors_sample[permutation]

        shuffled_output = model(
            center,
            shuffled_neighbors
        )

        shuffled_probability = torch.softmax(
            shuffled_output, dim=0
        )

        difference = torch.max(
            torch.abs(
                original_probability - shuffled_probability
            )
        ).item()

        differences.append(difference)

print("Permutation differences:")
print(differences)

print(
    "Maximum difference across all permutations:",
    max(differences)
)

Permutation differences:
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Maximum difference across all permutations: 0.0
